# Observability & Debugging — Part 5: Custom Metrics & Production Best Practices

This notebook teaches you how to add custom instrumentation for production monitoring
and apply best practices for sampling, performance, and security.

**What you'll learn:**
- Add custom span attributes for business context
- Use the OpenTelemetry Metrics API for aggregate monitoring
- Configure `BatchSpanProcessor` for production
- Apply sampling, alerting, and security best practices

**Prerequisites:**
- Complete notebooks 01–04 first

In [ ]:
import time

from strands import Agent, tool
from strands.models.bedrock import BedrockModel
from strands.telemetry.config import StrandsTelemetry
from opentelemetry import trace

# Configure telemetry
telemetry = StrandsTelemetry()
telemetry.setup_console_exporter()

print("✓ Telemetry configured")

## Custom Span Attributes

Add domain-specific metadata to spans using `trace.get_current_span().set_attribute()`.
This lets you correlate traces with business context: user IDs, session IDs,
feature flags, request types, etc.

**Where to add attributes:**
- Inside `@tool` functions → attributes appear on the tool span
- In a wrapper span around `agent()` → attributes appear on a parent span
- Both approaches are useful for different query patterns

In [ ]:
@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression with custom span attributes."""
    # Add custom attributes to the current tool span
    current_span = trace.get_current_span()
    current_span.set_attribute("app.expression", expression)
    current_span.set_attribute("app.tool_version", "1.0.0")

    try:
        result = eval(expression, {"__builtins__": {}}, {})
        current_span.set_attribute("app.result", str(result))
        current_span.set_attribute("app.success", True)
        return str(result)
    except Exception as e:
        current_span.set_attribute("app.success", False)
        current_span.set_attribute("app.error", str(e))
        return f"Error: {e}"


agent = Agent(
    model=BedrockModel(model_id="us.amazon.nova-lite-v1:0"),
    tools=[calculator],
)

print("✓ Agent created with instrumented calculator tool")

In [ ]:
# Add request-level context using a wrapper span
tracer = trace.get_tracer("observability-tutorial")

with tracer.start_as_current_span("user-request") as span:
    # Business context attributes
    span.set_attribute("app.user_id", "user-123")
    span.set_attribute("app.session_id", "session-abc")
    span.set_attribute("app.feature_flag", "calculator-v2")
    span.set_attribute("app.request_type", "math_query")

    result = agent("What is 99 * 77?")
    print(f"\nResult: {result}")

print("\n✓ Custom attributes added — visible in trace output above")
print("  Look for: app.user_id, app.session_id, app.expression, app.result")

## Custom Metrics

Beyond traces, OpenTelemetry supports **metrics** for aggregate monitoring.
Metrics answer questions like:
- How many invocations per minute?
- What's the P95 latency?
- What's the error rate over the last hour?
- How many tokens are we consuming per day?

In [ ]:
from opentelemetry import metrics
from opentelemetry.sdk.metrics import MeterProvider
from opentelemetry.sdk.metrics.export import (
    ConsoleMetricExporter,
    PeriodicExportingMetricReader,
)

# Set up metrics with console exporter
metric_reader = PeriodicExportingMetricReader(
    ConsoleMetricExporter(),
    export_interval_millis=10000,  # Export every 10 seconds
)
meter_provider = MeterProvider(metric_readers=[metric_reader])
metrics.set_meter_provider(meter_provider)

# Create custom instruments
meter = metrics.get_meter("strands-agent-metrics")

invocation_counter = meter.create_counter(
    name="agent.invocations",
    description="Number of agent invocations",
    unit="1",
)

latency_histogram = meter.create_histogram(
    name="agent.latency",
    description="Agent invocation latency",
    unit="ms",
)

error_counter = meter.create_counter(
    name="agent.errors",
    description="Number of failed agent invocations",
    unit="1",
)

print("✓ Metrics instruments created")
print("  - agent.invocations (counter)")
print("  - agent.latency (histogram)")
print("  - agent.errors (counter)")

In [ ]:
# Instrument an agent call with metrics
start = time.time()
try:
    result = agent("What is 2 + 2?")
    status = "success"
except Exception:
    status = "error"
    error_counter.add(1, {"agent.name": "calculator-agent"})

duration_ms = (time.time() - start) * 1000

# Record metrics
invocation_counter.add(1, {"agent.name": "calculator-agent", "status": status})
latency_histogram.record(duration_ms, {"agent.name": "calculator-agent"})

print(f"\nResult: {result}")
print(f"Latency: {duration_ms:.0f}ms")
print(f"Status: {status}")
print("\n✓ Metrics recorded (will export to console in ~10 seconds)")

## Production Configuration

For production, use `BatchSpanProcessor` instead of `SimpleSpanProcessor`.
It queues spans and exports them in bulk, adding minimal latency to agent calls.

| Setting | Development | Production |
|---------|-------------|------------|
| Span Processor | `SimpleSpanProcessor` | `BatchSpanProcessor` |
| Console Exporter | ✓ Enabled | ✗ Disabled |
| Sampling | 100% | 1–10% |
| Error Sampling | 100% | 100% (always) |

In [ ]:
# Install OTLP exporter (required for production config below)
!pip install opentelemetry-exporter-otlp-proto-http -q

In [ ]:
try:
    from opentelemetry.sdk.trace import TracerProvider
    from opentelemetry.sdk.trace.export import BatchSpanProcessor
    from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
    from opentelemetry.sdk.resources import Resource
    _OTLP_AVAILABLE = True
except ImportError:
    _OTLP_AVAILABLE = False
    print("⚠️  opentelemetry-exporter-otlp-proto-http not installed.")
    print("   Install with: pip install opentelemetry-exporter-otlp-proto-http")
    print("   The production config function below will not work without it.")


def setup_production_telemetry(endpoint: str, service_name: str = "my-agent-service"):
    """Configure telemetry for production use.

    Uses BatchSpanProcessor for minimal latency impact and
    OTLP export to the specified endpoint.

    Args:
        endpoint: OTLP endpoint URL (e.g., http://collector:4318)
        service_name: Service name for trace identification
    """
    if not _OTLP_AVAILABLE:
        print("✗ Cannot configure — install opentelemetry-exporter-otlp-proto-http first.")
        return

    resource = Resource.create({"service.name": service_name})
    provider = TracerProvider(resource=resource)

    # BatchSpanProcessor — queues spans and exports in bulk
    otlp_exporter = OTLPSpanExporter(endpoint=f"{endpoint}/v1/traces")
    provider.add_span_processor(
        BatchSpanProcessor(
            otlp_exporter,
            max_queue_size=2048,        # Buffer up to 2048 spans
            max_export_batch_size=512,  # Export 512 at a time
            schedule_delay_millis=5000, # Export every 5 seconds
        )
    )

    trace.set_tracer_provider(provider)
    print(f"✓ Production telemetry configured")
    print(f"  Endpoint: {endpoint}")
    print(f"  Service: {service_name}")
    print(f"  Processor: BatchSpanProcessor (queue=2048, batch=512, delay=5s)")


# Example (don't run unless you have a collector):
# setup_production_telemetry("http://localhost:4318", "my-agent")
if _OTLP_AVAILABLE:
    print("Production setup function defined.")

## Sampling Strategies

In production, tracing every request adds overhead. Use sampling:

```python
from opentelemetry.sdk.trace.sampling import TraceIdRatioBased, ParentBased

# Sample 10% of traces, but always trace if parent is sampled
sampler = ParentBased(root=TraceIdRatioBased(0.1))
provider = TracerProvider(sampler=sampler, resource=resource)
```

**Guidelines:**
- Development: 100% (see everything)
- Staging: 50–100%
- Production: 1–10% (adjust based on volume)
- **Always sample errors at 100%** regardless of rate

## Alert Thresholds

Set alerts on these signals for production agents:

| Signal | Threshold | Meaning |
|--------|-----------|--------|
| Tool error rate | > 5% over 5 min | Tool reliability degraded |
| P95 latency | > 30 seconds | Agent is slow |
| Context window usage | > 80% of limit | Risk of truncation |
| Cycle count | > 5 per invocation | Potential infinite loop |
| Token consumption | > budget threshold | Cost control |

## Telemetry Security

**Do:**
- Use TLS for OTLP export endpoints
- Rotate API keys for backend authentication
- Add generic metadata (user_id, session_id) to spans
- Redact PII before adding to attributes

**Don't:**
- Log raw user input in span attributes
- Include passwords, tokens, or secrets in traces
- Send traces to unencrypted endpoints in production
- Store full request/response bodies in span attributes

## Summary

You've completed the full observability tutorial! Here's what you've learned:

1. **Tracing setup** (01) — `StrandsTelemetry` with console exporter
2. **Trace hierarchy** (02) — Agent → Cycle → Model → Tool spans
3. **Debugging** (03) — Error spans, context pressure, token growth
4. **Backend export** (04) — OTLP to CloudWatch, Langfuse, Jaeger
5. **Production** (05) — Custom attributes, metrics, BatchSpanProcessor, sampling

### Key Takeaways

- `StrandsTelemetry` must be configured *before* creating `Agent` instances
- Filter spans by `status_code == ERROR` to find failures
- Monitor `gen_ai.usage.input_tokens` growth for context pressure
- Use `BatchSpanProcessor` in production
- Sample traces (1–10%) but always trace errors at 100%
- Never log sensitive data in span attributes

### Next Steps

- See [16-hooks-lifecycle](../16-hooks-lifecycle) for how hooks compose with tracing
- Explore multi-agent tracing with `Graph` or `Swarm` orchestration
- Set up dashboards in your chosen backend for ongoing monitoring
- Add custom ML-based anomaly detection on trace metrics